In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re
import pyodbc

In [ ]:
conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=prdsql05.westfund.com.au;'
    'DATABASE=BRONZE;'
    'Trusted_Connection=yes;'
)

In [ ]:
start_date = "20260601"
end_date   = "20260630"

table_prefix_list = [
    "person"
    # "person_membership"
]
# table_prefix_list = [
#     "memship",
#     "person"
# ]

In [ ]:
output_folder = r"\\prdeqs01\QlikData\bronze_snapshots_backup"

In [ ]:
cursor = conn.cursor()

In [ ]:
# =============================
# 循环每个前缀
# =============================
for table_prefix in table_prefix_list:

    print(f"\n=== Processing prefix: {table_prefix} ===")

    # 查找 snapshot 表
    query_tables = f"""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'dbo'
      AND TABLE_NAME LIKE '{table_prefix}_%' ESCAPE '\\'
    """
    tables = [row[0] for row in cursor.execute(query_tables).fetchall()]

    # 匹配表名中的日期
    pattern = re.compile(rf"{table_prefix}_(\d{{8}})")
    valid_tables = []

    # 过滤具体日期范围
    for t in tables:
        match = pattern.match(t)
        if match:
            full_date = match.group(1)  # YYYYMMDD
            if start_date <= full_date <= end_date:
                valid_tables.append((t, full_date))

    if not valid_tables:
        print(f"No snapshots found for prefix {table_prefix}.")
        continue

    print(f"Selected snapshot tables: {valid_tables}")

    # =============================
    # 读取每个 snapshot 表并增加 snapshot_date
    # =============================
    df_list = []
    for table_name, snapshot_date in valid_tables:
        print(f"Reading {table_name} ...")
        df = pd.read_sql(f"SELECT * FROM dbo.{table_name}", conn)
        df["snapshot_date"] = snapshot_date
        df_list.append(df)

    # =============================
    # 合并所有 snapshot 表
    # =============================
    df_all = pd.concat(df_list, ignore_index=True)

    # =============================
    # 输出 parquet 文件
    # =============================
    file_name = f"{table_prefix}_snapshot_{start_date}_{end_date}.parquet"
    output_path = os.path.join(output_folder, file_name)
    df_all.to_parquet(output_path, index=False)

    print(f"Saved parquet → {output_path}")

In [ ]:
# parquet 文件路径
file_name = "person_snapshot_20251101_20251130.parquet"
file_path = os.path.join(output_folder, file_name)

# 读取 parquet
df_person = pd.read_parquet(file_path)

# 筛选指定 snapshot_date
snapshot_date = "20251128"
df_snapshot = df_person[df_person["snapshot_date"] == snapshot_date]

In [ ]:
len(df_snapshot)

In [ ]:
info_list = []

for col in df_snapshot.columns:
    info_list.append({
        "column": col,
        "non_null_count": df_snapshot[col].notnull().sum(),
        "dtype": df_snapshot[col].dtype
    })

df_info = pd.DataFrame(info_list)

df_info


In [ ]:
df_snapshot[df_snapshot['membership_id'] == 58323]
# df_snapshot[df_snapshot['medicare_number'] == '4242956938']

In [ ]:
# DECLARE @startDate DATE = '2026-04-01';
# DECLARE @endDate   DATE = '2026-04-30';
# DECLARE @sql NVARCHAR(MAX) = N'';
# DECLARE @tableName NVARCHAR(128);
# DECLARE @currentDate DATE = @startDate;

# WHILE @currentDate <= @endDate
# BEGIN
#     SET @tableName = 'memship_' + FORMAT(@currentDate, 'yyyyMMdd');

#     -- 只有当表确实存在、且名字完全匹配时才加入删除列表
#     IF EXISTS (
#         SELECT 1 
#         FROM BRONZE.sys.tables t
#         JOIN BRONZE.sys.schemas s ON t.schema_id = s.schema_id
#         WHERE s.name = 'dbo' 
#           AND t.name = @tableName   -- 精确等值匹配，不是LIKE模糊匹配
#     )
#         SET @sql = @sql + 'DROP TABLE [BRONZE].[dbo].[' + @tableName + '];' + CHAR(13);

#     SET @currentDate = DATEADD(DAY, 1, @currentDate);
# END

# -- 第一步：先打印出所有将被删除的表，肉眼确认
# --PRINT @sql;
# EXEC sp_executesql @sql;